```{contents}
```

## Momentum 

### Concept and Intuition

**Momentum** is an optimization technique that accelerates gradient descent by accumulating a running average of past gradients.
It allows the optimizer to **build velocity in consistent directions** and **dampen oscillations** in noisy or curved loss landscapes.

Without momentum, gradient descent behaves like a person walking down a valley by only looking at the local slope.
With momentum, it behaves like a heavy ball rolling downhill: it keeps moving in the same direction unless strong opposing gradients act on it.

---

### Mathematical Formulation

Let:

* $\theta_t$ = model parameters at step $t$
* $g_t = \nabla_\theta \mathcal{L}(\theta_t)$ = gradient
* $v_t$ = velocity
* $\mu$ = momentum coefficient (typically 0.9)
* $\eta$ = learning rate

Update equations:

$$
v_t = \mu v_{t-1} + g_t
$$

$$
\theta_{t+1} = \theta_t - \eta v_t
$$

This causes gradients from previous steps to influence the current update.

---

### Why Momentum Works

| Problem in Vanilla GD   | Effect of Momentum   |
| ----------------------- | -------------------- |
| Slow convergence        | Speeds up learning   |
| Zig-zag in ravines      | Smooths oscillations |
| Sensitivity to noise    | Stabilizes updates   |
| Local minima & plateaus | Helps escape         |

---

### Visual Intuition

* **Steep ravine** → vanilla GD bounces side-to-side
* **Momentum** → averages direction → smooth downhill path

---

### Training Workflow with Momentum

1. Compute gradient on current batch
2. Update velocity using historical gradients
3. Update parameters using velocity
4. Repeat

---

### PyTorch Demonstration

#### Dataset Setup



In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=2000, noise=0.15)
X = StandardScaler().fit_transform(X)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)




---

#### Model Definition



In [2]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2))

    def forward(self, x):
        return self.net(x)




---

#### Training Without Momentum



In [3]:
model = Net()
opt = optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

for _ in range(50):
    opt.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()




---

#### Training With Momentum



In [4]:
model_m = Net()
opt_m = optim.SGD(model_m.parameters(), lr=0.05, momentum=0.9)

for _ in range(50):
    opt_m.zero_grad()
    loss = loss_fn(model_m(X_train), y_train)
    loss.backward()
    opt_m.step()




---

### Observed Effects

| Metric             | Without Momentum | With Momentum |
| ------------------ | ---------------- | ------------- |
| Convergence speed  | Slow             | Fast          |
| Training stability | Noisy            | Smooth        |
| Final accuracy     | Lower            | Higher        |
| Sensitivity to LR  | High             | Lower         |

---

### Variants of Momentum

| Method             | Description                           |
| ------------------ | ------------------------------------- |
| Classical Momentum | Uses past gradient average            |
| Nesterov Momentum  | Looks ahead before computing gradient |
| Adam               | Momentum + adaptive learning rate     |
| RMSProp            | Momentum on squared gradients         |

---

### Nesterov Momentum (Intuition)

Instead of:
$$
g_t = \nabla \mathcal{L}(\theta_t)
$$

Compute:
$$
g_t = \nabla \mathcal{L}(\theta_t - \mu v_{t-1})
$$

This anticipates where parameters are going, providing more accurate updates.

---

### When to Use Momentum

| Scenario              | Recommendation       |
| --------------------- | -------------------- |
| Deep networks         | Strongly recommended |
| Noisy gradients       | Essential            |
| Complex loss surfaces | Highly effective     |
| Small batch training  | Improves stability   |

---

### Summary

| Property          | Effect                        |
| ----------------- | ----------------------------- |
| Purpose           | Accelerate convergence        |
| Core idea         | Accumulate gradient history   |
| Primary benefit   | Faster, smoother optimization |
| Modern optimizers | All use momentum principles   |

---

### Key Insight

> **Momentum transforms gradient descent from a short-sighted walker into a fast, stable optimizer with memory.**
